# LoRA od środka: wstępne zrozumienie projektu
Notebook na konsultacje wstępne (temat 14, GSN 26Z).

## Cel
Zrozumieć PEFT przez samodzielną implementację LoRA, a nie tylko użycie gotowej biblioteki.

## Minimalny zakres
- Mały pretrenowany Transformer, np. OPT-125M.
- Własny `LoRALinear`: zamrożone `W` + trenowalne `A` i `B`.
- Wpięcie LoRA do `q_proj` / `v_proj`.
- Fine-tuning na małym zadaniu tekstowym.

**Baseline:** mały Transformer / OPT-125M. **Dane:** kilka do kilkunastu tysięcy krótkich tekstów.

## Co sprawdzić
- head-only vs własna LoRA
- różne rangi `r`
- różne warstwy z wstawioną LoRA
- jakość + liczba trenowanych parametrów + VRAM

## Dlaczego ten mechanizm?
LoRA zamraża dużą macierz `W` i uczy tylko małą niskorangową poprawkę `BA`, ograniczając liczbę trenowanych parametrów.

$$h = Wx + \frac{\alpha}{r} B A x,\quad W\in\mathbb{R}^{d_{out}\times d_{in}},\ A\in\mathbb{R}^{r\times d_{in}},\ B\in\mathbb{R}^{d_{out}\times r}$$

- `B = 0` na starcie, więc `ΔW = 0` i model startuje jak pretrenowany.
- `A` ma małą losową inicjalizację (przy samych zerach gradient byłby zerowy).
- Uczymy tylko `A` i `B`.

## Rachunek parametrów
Pełna macierz: `d_in * d_out`. LoRA: `r * (d_in + d_out)`.

In [ ]:
d = 768  # hidden size OPT-125M
for r in [1, 2, 4, 8, 16, 32]:
    full = d * d
    lora = r * (d + d)
    print(f'r={r:>2}  LoRA={lora:>7,}  pełna={full:,}  ({100*lora/full:.2f}%)')

## Struktura OPT-125M
Znajdź `q_proj` / `v_proj`, sprawdź kształty.

In [ ]:
from transformers import AutoModel, AutoTokenizer
name = 'facebook/opt-125m'
tok = AutoTokenizer.from_pretrained(name)
model = AutoModel.from_pretrained(name)
print(model.decoder.layers[0])
sum(p.numel() for p in model.parameters())

In [ ]:
batch = tok(['to jest test', 'LoRA is neat'], padding=True, return_tensors='pt')
out = model(**batch).last_hidden_state
print({k: tuple(v.shape) for k, v in batch.items()}, '->', tuple(out.shape))  # [B,T] -> [B,T,d]

## Szkic implementacji (z prezentacji projektowej)
Szkielet do uzupełnienia jest w `src/lora_project/lora.py`.

```python
class LoRALinear(nn.Module):
    def __init__(self, base_linear, rank=4, alpha=8):
        ...  # zamroź W, A: randn*0.01 [r,in], B: zeros [out,r], scale=alpha/r
    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)

for block in model.model.decoder.layers:
    block.self_attn.q_proj = LoRALinear(block.self_attn.q_proj)
    block.self_attn.v_proj = LoRALinear(block.self_attn.v_proj)
```

## Plan eksperymentów
| Eksperyment | Zmienna | Mierzymy |
|---|---|---|
| head-only | brak | jakość, parametry, VRAM |
| LoRA q,v | `r` ∈ {1,2,4,8,16,32} | jakość, parametry, VRAM, czas |
| LoRA warstwy | tylko q / q+v / wszystkie | jakość, parametry |

## Otwarte pytania
Zob. `README.md`: zadanie i dane, zakres baseline'u, użycie `peft`, wymagane rangi/warstwy, sprzęt, metryki, format oddania.